In [ ]:
# 0) Đường dẫn dữ liệu
from pathlib import Path
DATA_PATH = Path('/mnt/data/data.csv')
assert DATA_PATH.exists(), f'Không tìm thấy file: {DATA_PATH}'
DATA_PATH

In [ ]:
# 1) Load dữ liệu & xác định cột nhãn
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
display(df.head())

# tìm cột nhãn theo danh sách ưu tiên (case‑insensitive)
cands = ['label','class','species','target','y','dangerous']
lower = {c.lower(): c for c in df.columns}
target_col = None
for k in cands:
    if k in lower:
        target_col = lower[k]
        break
if target_col is None:
    target_col = df.columns[-1]  # fallback

y_raw = df[target_col]
if y_raw.dtype == object:
    cat = pd.Categorical(y_raw)
    y = cat.codes  # 0..K-1
    classes_ = list(cat.categories.astype(str))
else:
    y = y_raw.astype(int).values
    classes_ = sorted(list(map(str, np.unique(y))))
n_classes = len(classes_)

print('Target column:', target_col)
print('Classes:', classes_)
print('n_classes:', n_classes)

In [ ]:
# 2) One‑hot features
X = pd.get_dummies(df.drop(columns=[target_col]), drop_first=False)
feature_names = X.columns.tolist()
X.shape, len(feature_names)

In [ ]:
# 3) Chia train/test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

In [ ]:
# 4) Pipeline SVM (Linear) + huấn luyện
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False)),
    ('svm', SVC(kernel='linear', probability=True, class_weight='balanced', max_iter=3000, random_state=42))
])

svm_pipe.fit(X_train, y_train)
svm_pipe

In [ ]:
# 5) Đánh giá mô hình
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

y_pred = svm_pipe.predict(X_test)
y_proba = svm_pipe.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
prec_weight = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec_weight = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1_weight = f1_score(y_test, y_pred, average='weighted', zero_division=0)

try:
    if n_classes == 2:
        rocauc = roc_auc_score(y_test, y_proba[:,1])
    else:
        rocauc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
except Exception:
    rocauc = None

print('Accuracy        :', acc)
print('Precision (macro):', prec_macro)
print('Recall (macro)   :', rec_macro)
print('F1 (macro)       :', f1_macro)
print('Precision (w)    :', prec_weight)
print('Recall (w)       :', rec_weight)
print('F1 (w)           :', f1_weight)
print('ROC-AUC          :', rocauc)
print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=classes_, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
# 6) Top đặc trưng theo |coef| của Linear SVM (One‑vs‑Rest)
import numpy as np

# Lấy hệ số từ bước cuối của pipeline
clf = svm_pipe.named_steps['svm']
coefs = clf.coef_
if coefs.ndim == 1:
    coefs = coefs.reshape(1, -1)

# Tổng hợp độ quan trọng: trung bình trị tuyệt đối qua các lớp
importance = np.mean(np.abs(coefs), axis=0)
topk = 20 if len(importance) >= 20 else len(importance)
top_idx = np.argsort(importance)[::-1][:topk]
top_features = [(feature_names[i], float(importance[i])) for i in top_idx]

import pandas as pd
top_df = pd.DataFrame(top_features, columns=['feature','importance(|coef|)'])
display(top_df)
top_df.head(10)

In [ ]:
# 7) Lưu artifacts: model + summary + confusion matrix
from joblib import dump
import json

MODEL_PATH = '/mnt/data/forest_animals_svm.joblib'
CM_PATH = '/mnt/data/forest_animals_confusion_matrix.csv'
SUMMARY_PATH = '/mnt/data/forest_animals_svm_summary.json'
TOPFEAT_PATH = '/mnt/data/forest_animals_top_features.csv'

dump(svm_pipe, MODEL_PATH)
pd.DataFrame(cm).to_csv(CM_PATH, index=False)
top_df.to_csv(TOPFEAT_PATH, index=False)

summary = {
    'dataset': str(DATA_PATH),
    'target_col': target_col,
    'classes': classes_,
    'n_classes': n_classes,
    'X_shape': {'train': list(X_train.shape), 'test': list(X_test.shape)},
    'metrics': {
        'accuracy': float(acc),
        'precision_macro': float(prec_macro),
        'recall_macro': float(rec_macro),
        'f1_macro': float(f1_macro),
        'precision_weighted': float(prec_weight),
        'recall_weighted': float(rec_weight),
        'f1_weighted': float(f1_weight),
        'roc_auc_macro_ovr': (None if rocauc is None else float(rocauc))
    },
    'artifacts': {
        'model_joblib': MODEL_PATH,
        'confusion_matrix_csv': CM_PATH,
        'top_features_csv': TOPFEAT_PATH
    }
}
with open(SUMMARY_PATH,'w',encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

SUMMARY_PATH